In [ ]:
import os
import streamlit as st
import pickle
import langchain

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_classic.chains.qa_with_sources.retrieval import (
    RetrievalQAWithSourcesChain
)

from langchain_classic.chains.qa_with_sources.loading import (
    load_qa_with_sources_chain
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import UnstructuredURLLoader

from langchain_community.vectorstores import FAISS

In [ ]:
import os
from dotenv import load_dotenv

# 1. Load variables from .env
load_dotenv()

# 2. Read the variable safely using .get() to prevent KeyError if missing
api_key = os.getenv("GOOGLE_API_KEY")


In [ ]:
### Initialized LLM with required params
llm = ChatGoogleGenerativeAI(
    model = "models/gemini-3.1-flash-lite",
    temperature=0.9,
    max_tokens=500,
    google_api_key=api_key
)

loader = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])

data = loader.load()
len(data)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)

### As data is of type documents we can directly use split_documents over  aplit_text in order to get the chunks.
docs = text_splitter.split_documents(data)
len(data)

In [ ]:
docs[9]

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

vectorindex_gemini = FAISS.from_documents(
    docs,
    embeddings
)

In [ ]:
file_path = "vector_index.pkl"

data_to_save = {
    "index": vectorindex_gemini.index,
    "docstore": vectorindex_gemini.docstore,
    "index_to_docstore_id": vectorindex_gemini.index_to_docstore_id
}

with open(file_path, "wb") as f:
    pickle.dump(data_to_save, f)

print("Vector index saved to", file_path)

In [ ]:
if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        loaded_data = pickle.load(f)

    # Reconstruct the FAISS vector store
    vectorIndex = FAISS(
        embedding_function=embeddings,
        index=loaded_data["index"],
        docstore=loaded_data["docstore"],
        index_to_docstore_id=loaded_data["index_to_docstore_id"]
    )

In [ ]:
chain = RetrievalQAWithSourcesChain.from_llm(
    llm=llm,
    retriever = vectorIndex.as_retriever()
)

# chain

In [ ]:
query = "What is the price of tiago iCNG"

langchain.debug = True

chain({"question": query}, return_only_outputs = True )